# Method 2 — BERT Token Classification (text only)

Issue #5 (Hiranya): fine-tune bert-base-uncased on SROIE.

In [ ]:
!pip install -q transformers "datasets<3" seqeval

In [ ]:
# Make `src/` importable. In a fresh Colab runtime the repo isn't there yet, so clone it.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/chundawat-h/Receipt-Invoice-Information-Extraction.git"
if Path("../src").is_dir():            # running from notebooks/ inside the repo
    ROOT = Path("..")
elif Path("src").is_dir():
    ROOT = Path(".")
else:                                  # Colab
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    ROOT = Path("Receipt-Invoice-Information-Extraction")
sys.path.insert(0, str(ROOT.resolve()))
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

### Notes from data exploration (#4)
- Only 4 train / 1 test receipts exceed 512 sub-tokens, and the words cut off are never field words, so `truncation=True, max_length=512` is fine. Give cut-off words the label `"O"` before calling `bio_to_fields`, otherwise it raises a length mismatch.
- Evaluate on the 347 test receipts and save `results/bert_preds.json` keyed by `row["key"]`.
- Don't use the test split for checkpoint selection (no `load_best_model_at_end` on test). Train the fixed 10 epochs, then evaluate once.

In [ ]:
from src.data import LABEL2ID, LABELS, ID2LABEL, load_sroie

# Word-tagged SROIE (darentang/sroie, Parquet copy): train 626 / test 347.
# Columns: id, words, bboxes (already 0-1000), ner_tags, image_path, key.
# Save predictions keyed by row["key"] (e.g. "X51005230621"), not by row index.
ds = load_sroie()
ds